# dow-ranker quickstart

Install: `pip install "dow-ranker[app] @ git+https://github.com/OWNER/dow-ranker@v0.1.0"`, then `dow-ranker fetch-data` for the UCI data.

In [1]:
import pandas as pd
import dow_ranker as dr
from dow_ranker.synthetic import make_panel
pd.set_option('display.width', 140)
print(dr.__version__)

0.1.0


## 1. Your own data: one row per stock and week
Here a synthetic panel in the required format (4 weeks x 30 stocks).

In [2]:
panel = make_panel(seed=1)
panel.head()

,stock,date,open,high,low,close,volume
0,AA,2011-04-01,65.838198,69.254005,64.679757,68.726285,35440878.0
1,AXP,2011-04-01,96.611458,97.668759,93.104627,94.608630,26658562.0
2,BA,2011-04-01,39.973042,40.176399,39.315315,39.731507,24287980.0
3,BAC,2011-04-01,96.085191,101.050984,94.958501,100.155849,25055726.0
4,CAT,2011-04-01,51.859234,53.203686,51.313909,52.608274,27745906.0


In [3]:
r = dr.predict_week(panel, level=0.8)
print('week', r.week.date(), '| warnings:', r.warnings)
r.table[['rank', 'stock', 'pred_rel_return', 'lower', 'upper', 'out_of_range']].head(10)

week 2011-04-22 | warnings: ['features outside the training range (prediction is an extrapolation): log_dollar_vol (1 stocks)']


,rank,stock,pred_rel_return,lower,upper,out_of_range
0,1,GE,0.169223,-2.006822,2.345267,log_dollar_vol
1,2,IBM,0.127545,-2.048500,2.303589,
2,3,DD,0.099385,-2.076659,2.275430,
3,4,KO,0.091391,-2.084654,2.267435,
4,5,JNJ,0.078797,-2.097247,2.254842,
5,6,KRFT,0.072167,-2.103878,2.248211,
6,7,VZ,0.068429,-2.107615,2.244474,
7,8,AA,0.055718,-2.120327,2.231762,
8,9,DIS,0.044802,-2.131243,2.220846,
9,10,MCD,0.044522,-2.131522,2.220566,


## 2. The UCI data: rank a held-out week (Q2 2011, never used in training)

In [4]:
uci = dr.fetch_uci('../data') if not __import__('pathlib').Path('../data/dow_jones_index.data').exists() else '../data/dow_jones_index.data'
r = dr.predict_week(uci, week='2011-05-13')
for w in r.warnings: print('WARNING:', w)
r.table[['rank', 'stock', 'pred_rel_return', 'lower', 'upper']].head(10)

,rank,stock,pred_rel_return,lower,upper
0,1,TRV,0.133147,-2.042898,2.309191
1,2,BA,0.115151,-2.060893,2.291196
2,3,VZ,0.115062,-2.060982,2.291106
3,4,HD,0.112704,-2.063340,2.288749
4,5,UTX,0.108172,-2.067872,2.284217
5,6,MMM,0.099647,-2.076397,2.275692
6,7,AXP,0.075546,-2.100498,2.251591
7,8,DD,0.066589,-2.109456,2.242633
8,9,T,0.057210,-2.118834,2.233255
9,10,AA,0.056460,-2.119584,2.232505


## 3. Does it work? Coverage, error and rank correlation on the held-out weeks

In [5]:
dr.backtest_summary(dr.backtest(uci, level=0.8, start='2011-04-01')).round(3)

,n,mae,mae_baseline_zero,coverage,ic
date,,,,,
2011-04-01,30,1.140,1.126,0.767,-0.236
2011-04-08,30,2.017,2.050,0.533,0.592
2011-04-15,30,1.921,1.934,0.733,0.085
2011-04-21,30,1.712,1.695,0.667,-0.166
2011-04-29,30,1.472,1.464,0.800,0.203
2011-05-06,30,1.826,1.831,0.567,0.136
2011-05-13,30,1.482,1.497,0.833,0.067
2011-05-20,30,1.290,1.269,0.800,-0.321
2011-05-27,30,1.145,1.140,0.900,0.028


## 4. Bad input is refused or flagged, not silently ranked

In [6]:
bad = panel.drop(columns='volume')
try:
    dr.predict_week(bad)
except dr.InputError as e:
    print('InputError:', e)

odd = panel.copy()
odd.loc[odd.index[-1], 'volume'] *= 1000      # a volume spike far outside 2011
print(dr.predict_week(odd).warnings)

InputError: missing required column(s): volume; needed: stock, date, open, high, low, close, volume
['features outside the training range (prediction is an extrapolation): log_vol_ratio (1 stocks), log_dollar_vol (2 stocks)']
